In [17]:
import pickle
import numpy as np
import xgboost as xgb
from scipy.stats import spearmanr

In [18]:
# Dataset: daily CSI300/CSI800, 2008–2022.
# Split (paper): train Q1 2008–Q1 2020 | valid Q2 2020 | test Q3 2020–Q4 2022 (last 10 quarters).
market = 'HK_market' # ['HK_market', 'US_market']
if market == 'HK_market':
    universe = 'csi300' # ['csi300','csi800']
    train_data_dir = 'data'
    with open(f'data/HK_market/{universe}_dl_train.pkl', 'rb') as f:
        dl_train = pickle.load(f)
    with open(f'data/HK_market/{universe}_dl_valid.pkl', 'rb') as f:
        dl_valid = pickle.load(f)
    with open(f'data/HK_market/{universe}_dl_test.pkl', 'rb') as f:
        dl_test = pickle.load(f) # class 'qlib.data.dataset.TSDataSampler'

In [ ]:
# TSDataSampler -> (X, y) arrays. Use index 0..len-1 (qlib iterator can yield one past end).
def sampler_to_arrays(sampler):
    X_list, y_list = [], []
    n = len(sampler)
    for i in range(n):
        batch = sampler[i]
        if isinstance(batch, (list, tuple)):
            x, y = batch[0], batch[1]
        elif isinstance(batch, np.ndarray):
            if batch.size == 2 and batch.ndim == 1:
                x, y = batch[0], batch[1]
            elif batch.ndim == 2:
                x, y = batch[:, :-1], batch[:, -1]
            else:
                x = batch[..., :-1].reshape(batch.shape[0], -1)
                y = batch[..., -1].ravel()
        else:
            x = batch.get('feature', batch.get('x'))
            y = batch.get('label', batch.get('y'))
        x = np.asarray(x)
        y = np.asarray(y).ravel() if hasattr(y, 'ravel') else np.asarray(y)
        if x.ndim == 3:
            x = x.reshape(x.shape[0], -1)
        X_list.append(x)
        y_list.append(y)
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)

X_train, y_train = sampler_to_arrays(dl_train)
X_valid, y_valid = sampler_to_arrays(dl_valid)
X_test, y_test = sampler_to_arrays(dl_test)
print('Shapes: train', X_train.shape, 'valid', X_valid.shape, 'test', X_test.shape)


In [ ]:
# XGBoost: train on train, use valid for early stopping / eval, then test.
model = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    objective='reg:squarederror',
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False,
)
model.save_model('xgb_model.json')


Train: MSE = 0.001855, IC = 0.6336
Valid: MSE = 0.002013, IC = 0.0356
Test: MSE = nan, IC = nan


In [ ]:
# Predict on train, valid, test and report MSE + IC (skip NaN/Inf so Test metrics are valid)
def mse_ic(y_true, y_pred):
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    n_ok, n_tot = int(ok.sum()), len(ok)
    if n_ok == 0:
        return np.nan, np.nan, n_ok, n_tot
    y_true, y_pred = y_true[ok], y_pred[ok]
    mse = np.mean((y_true - y_pred) ** 2)
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        ic = np.nan  # spearmanr undefined for constant array
    else:
        ic, _ = spearmanr(y_true, y_pred)
    return mse, ic, n_ok, n_tot

for name, X, y in [('Train', X_train, y_train), ('Valid', X_valid, y_valid), ('Test', X_test, y_test)]:
    pred = model.predict(X)
    mse, ic, n_ok, n_tot = mse_ic(y, pred)
    print(f'{name}: MSE = {mse:.6f}, IC = {ic:.4f}  (n={n_ok}/{n_tot} valid)')
print()

Train: MSE = 0.001855, IC = 0.6336  (n=6849968/6849968 valid)
Valid: MSE = 0.002013, IC = 0.0356  (n=160312/160312 valid)
Test: MSE = 0.002987, IC = 0.0017  (n=1483160/1485600 valid)
  -> y_test: 2440 NaN, 0 Inf  |  pred: 0 NaN, 0 Inf

